In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

PROJECT_DIR = Path.cwd().parent
DATA_FILE = PROJECT_DIR / "data" / "pm25_forecasting_dataset.csv"

feature_df = pd.read_csv(DATA_FILE)

feature_df["observed_at_ist"] = pd.to_datetime(
    feature_df["observed_at_ist"]
)

feature_df = feature_df.sort_values(
    ["station_name", "observed_at_ist"]
).reset_index(drop=True)

print("Dataset loaded successfully!")
print("Shape:", feature_df.shape)
print("Stations:", feature_df["station_name"].nunique())

display(feature_df.head())

Dataset loaded successfully!
Shape: (197697, 41)
Stations: 13


,observed_at_ist,observed_at,location_id,station_name,candidate_station,region,latitude,longitude,co,no,...,wind_speed_missing,wind_direction_sin,wind_direction_cos,PM25_next_1h,PM25_next_3h,available_aqi_pollutants,pm25_24h_avg,pm25_aqi_category,target_1h,target_3h
0,2025-02-19 01:00:00+05:30,2025-02-18 19:30:00+00:00,6932,"Alipur, Delhi - DPCC",Alipur,North Delhi,28.815329,77.15301,0.150,5.0,...,0,-0.766044,-0.642788,59.8,146.0,6,NaN,NaN,NaN,NaN
1,2025-02-19 02:00:00+05:30,2025-02-18 20:30:00+00:00,6932,"Alipur, Delhi - DPCC",Alipur,North Delhi,28.815329,77.15301,0.350,16.3,...,0,-0.719340,-0.694658,82.0,205.0,6,NaN,NaN,NaN,NaN
2,2025-02-19 03:00:00+05:30,2025-02-18 21:30:00+00:00,6932,"Alipur, Delhi - DPCC",Alipur,North Delhi,28.815329,77.15301,0.645,23.5,...,0,-0.309017,-0.951057,146.0,174.0,6,NaN,NaN,NaN,NaN
3,2025-02-19 04:00:00+05:30,2025-02-18 22:30:00+00:00,6932,"Alipur, Delhi - DPCC",Alipur,North Delhi,28.815329,77.15301,0.588,12.7,...,0,-0.544639,-0.838671,205.0,128.0,6,NaN,NaN,NaN,NaN
4,2025-02-19 05:00:00+05:30,2025-02-18 23:30:00+00:00,6932,"Alipur, Delhi - DPCC",Alipur,North Delhi,28.815329,77.15301,0.563,11.5,...,0,-0.809017,-0.587785,174.0,119.0,6,NaN,NaN,NaN,NaN


In [2]:
# Extract time-based features
feature_df["hour"] = feature_df["observed_at_ist"].dt.hour

feature_df["day"] = feature_df["observed_at_ist"].dt.day

feature_df["day_of_week"] = (
    feature_df["observed_at_ist"].dt.dayofweek
)

feature_df["month"] = feature_df["observed_at_ist"].dt.month

feature_df["is_weekend"] = (
    feature_df["day_of_week"] >= 5
).astype(int)

# Cyclical encoding for hour
feature_df["hour_sin"] = np.sin(
    2 * np.pi * feature_df["hour"] / 24
)

feature_df["hour_cos"] = np.cos(
    2 * np.pi * feature_df["hour"] / 24
)

# Cyclical encoding for month
feature_df["month_sin"] = np.sin(
    2 * np.pi * (feature_df["month"] - 1) / 12
)

feature_df["month_cos"] = np.cos(
    2 * np.pi * (feature_df["month"] - 1) / 12
)

print("Time-based features created!")

display(
    feature_df[
        [
            "observed_at_ist",
            "hour",
            "day_of_week",
            "month",
            "is_weekend",
            "hour_sin",
            "hour_cos",
            "month_sin",
            "month_cos"
        ]
    ].head(10)
)

Time-based features created!


,observed_at_ist,hour,day_of_week,month,is_weekend,hour_sin,hour_cos,month_sin,month_cos
0,2025-02-19 01:00:00+05:30,1,2,2,0,0.258819,9.659258e-01,0.5,0.866025
1,2025-02-19 02:00:00+05:30,2,2,2,0,0.500000,8.660254e-01,0.5,0.866025
2,2025-02-19 03:00:00+05:30,3,2,2,0,0.707107,7.071068e-01,0.5,0.866025
3,2025-02-19 04:00:00+05:30,4,2,2,0,0.866025,5.000000e-01,0.5,0.866025
4,2025-02-19 05:00:00+05:30,5,2,2,0,0.965926,2.588190e-01,0.5,0.866025
5,2025-02-19 06:00:00+05:30,6,2,2,0,1.000000,6.123234e-17,0.5,0.866025
6,2025-02-19 07:00:00+05:30,7,2,2,0,0.965926,-2.588190e-01,0.5,0.866025
7,2025-02-19 08:00:00+05:30,8,2,2,0,0.866025,-5.000000e-01,0.5,0.866025
8,2025-02-19 09:00:00+05:30,9,2,2,0,0.707107,-7.071068e-01,0.5,0.866025
9,2025-02-19 10:00:00+05:30,10,2,2,0,0.500000,-8.660254e-01,0.5,0.866025


In [3]:
# Create historical PM2.5 lag features
lag_periods = [1, 3, 6, 12, 24]

for lag in lag_periods:
    feature_df[f"pm25_lag_{lag}"] = (
        feature_df.groupby("station_name")["pm25"].shift(lag)
    )

print("Lag features created!")

display(
    feature_df[
        [
            "station_name",
            "observed_at_ist",
            "pm25",
            "pm25_lag_1",
            "pm25_lag_3",
            "pm25_lag_6",
            "pm25_lag_12",
            "pm25_lag_24"
        ]
    ].head(30)
)

Lag features created!


,station_name,observed_at_ist,pm25,pm25_lag_1,pm25_lag_3,pm25_lag_6,pm25_lag_12,pm25_lag_24
0,"Alipur, Delhi - DPCC",2025-02-19 01:00:00+05:30,60.5,NaN,NaN,NaN,NaN,NaN
1,"Alipur, Delhi - DPCC",2025-02-19 02:00:00+05:30,59.8,60.5,NaN,NaN,NaN,NaN
2,"Alipur, Delhi - DPCC",2025-02-19 03:00:00+05:30,82.0,59.8,NaN,NaN,NaN,NaN
3,"Alipur, Delhi - DPCC",2025-02-19 04:00:00+05:30,146.0,82.0,60.5,NaN,NaN,NaN
4,"Alipur, Delhi - DPCC",2025-02-19 05:00:00+05:30,205.0,146.0,59.8,NaN,NaN,NaN
5,"Alipur, Delhi - DPCC",2025-02-19 06:00:00+05:30,174.0,205.0,82.0,NaN,NaN,NaN
6,"Alipur, Delhi - DPCC",2025-02-19 07:00:00+05:30,128.0,174.0,146.0,60.5,NaN,NaN
7,"Alipur, Delhi - DPCC",2025-02-19 08:00:00+05:30,119.0,128.0,205.0,59.8,NaN,NaN
8,"Alipur, Delhi - DPCC",2025-02-19 09:00:00+05:30,108.0,119.0,174.0,82.0,NaN,NaN
9,"Alipur, Delhi - DPCC",2025-02-19 10:00:00+05:30,90.8,108.0,128.0,146.0,NaN,NaN


In [4]:
grouped_pm25 = feature_df.groupby("station_name")["pm25"]

feature_df["pm25_rolling_mean_6h"] = (
    grouped_pm25.transform(
        lambda s: s.shift(1).rolling(6, min_periods=4).mean()
    )
)

feature_df["pm25_rolling_mean_12h"] = (
    grouped_pm25.transform(
        lambda s: s.shift(1).rolling(12, min_periods=8).mean()
    )
)

feature_df["pm25_rolling_mean_24h"] = (
    grouped_pm25.transform(
        lambda s: s.shift(1).rolling(24, min_periods=16).mean()
    )
)

feature_df["pm25_rolling_std_24h"] = (
    grouped_pm25.transform(
        lambda s: s.shift(1).rolling(24, min_periods=16).std()
    )
)

print("Rolling features created!")

display(
    feature_df[
        [
            "pm25",
            "pm25_lag_1",
            "pm25_rolling_mean_6h",
            "pm25_rolling_mean_12h",
            "pm25_rolling_mean_24h",
            "pm25_rolling_std_24h"
        ]
    ].head(35)
)

Rolling features created!


,pm25,pm25_lag_1,pm25_rolling_mean_6h,pm25_rolling_mean_12h,pm25_rolling_mean_24h,pm25_rolling_std_24h
0,60.5,NaN,NaN,NaN,NaN,NaN
1,59.8,60.5,NaN,NaN,NaN,NaN
2,82.0,59.8,NaN,NaN,NaN,NaN
3,146.0,82.0,NaN,NaN,NaN,NaN
4,205.0,146.0,87.075000,NaN,NaN,NaN
5,174.0,205.0,110.660000,NaN,NaN,NaN
6,128.0,174.0,121.216667,NaN,NaN,NaN
7,119.0,128.0,132.466667,NaN,NaN,NaN
8,108.0,119.0,142.333333,121.787500,NaN,NaN
9,90.8,108.0,146.666667,120.255556,NaN,NaN


In [5]:
# Check timestamp continuity within each station
feature_df["time_difference"] = (
    feature_df.groupby("station_name")["observed_at_ist"]
    .diff()
)

print("Time difference distribution:")
display(feature_df["time_difference"].value_counts(dropna=False))

print(
    "\nNon-hourly intervals:",
    (feature_df["time_difference"].notna() &
     (feature_df["time_difference"] != pd.Timedelta(hours=1))).sum()
)

Time difference distribution:


time_difference
0 days 01:00:00    197684
NaT                    13
Name: count, dtype: int64


Non-hourly intervals: 0


In [6]:
new_features = [
    "hour", "day", "day_of_week", "month", "is_weekend",
    "hour_sin", "hour_cos", "month_sin", "month_cos",
    "pm25_lag_1", "pm25_lag_3", "pm25_lag_6",
    "pm25_lag_12", "pm25_lag_24",
    "pm25_rolling_mean_6h",
    "pm25_rolling_mean_12h",
    "pm25_rolling_mean_24h",
    "pm25_rolling_std_24h"
]

print("Number of new features:", len(new_features))
print("Total dataset columns:", len(feature_df.columns))

display(feature_df[new_features].describe().T)

Number of new features: 18
Total dataset columns: 60


,count,mean,std,min,25%,50%,75%,max
hour,197697.0,11.496937,6.920923,0.0,5.000000,1.100000e+01,17.000000,23.000000
day,197697.0,15.699313,8.817100,1.0,8.000000,1.600000e+01,23.000000,31.000000
day_of_week,197697.0,3.002261,1.996111,0.0,1.000000,3.000000e+00,5.000000,6.000000
month,197697.0,6.542345,3.104291,1.0,4.000000,7.000000e+00,9.000000,12.000000
is_weekend,197697.0,0.285164,0.451493,0.0,0.000000,0.000000e+00,1.000000,1.000000
hour_sin,197697.0,0.000345,0.707101,-1.0,-0.707107,1.224647e-16,0.707107,1.000000
hour_cos,197697.0,-0.000245,0.707116,-1.0,-0.707107,-1.836970e-16,0.707107,1.000000
month_sin,197697.0,0.028506,0.732509,-1.0,-0.866025,1.224647e-16,0.866025,1.000000
month_cos,197697.0,-0.134294,0.666774,-1.0,-0.866025,-1.836970e-16,0.500000,1.000000
pm25_lag_1,176462.0,88.754525,87.325663,0.0,33.300000,5.750000e+01,112.000000,997.000000


In [7]:
# Remove temporary diagnostic column
feature_df.drop(
    columns=["time_difference"],
    errors="ignore",
    inplace=True
)

# Columns that must never be used as model inputs
excluded_from_features = [
    "observed_at",
    "observed_at_ist",
    "location_id",
    "station_name",
    "region",
    "latitude",
    "longitude",
    "candidate_station",  # retained separately for encoding
    "PM25_next_1h",
    "PM25_next_3h",
    "target_1h",
    "target_3h"
]

# Future target columns remain in the dataset,
# but will be excluded from model inputs.
print("Feature engineering completed!")
print("Final dataset shape:", feature_df.shape)

print("\nNew engineered features:")
print(new_features)

print("\nTarget columns:")
print(["target_1h", "target_3h"])

print("\nExcluded future leakage columns:")
print(["PM25_next_1h", "PM25_next_3h"])

Feature engineering completed!
Final dataset shape: (197697, 59)

New engineered features:
['hour', 'day', 'day_of_week', 'month', 'is_weekend', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'pm25_lag_1', 'pm25_lag_3', 'pm25_lag_6', 'pm25_lag_12', 'pm25_lag_24', 'pm25_rolling_mean_6h', 'pm25_rolling_mean_12h', 'pm25_rolling_mean_24h', 'pm25_rolling_std_24h']

Target columns:
['target_1h', 'target_3h']

Excluded future leakage columns:
['PM25_next_1h', 'PM25_next_3h']


In [8]:
OUTPUT_FILE = PROJECT_DIR / "data" / "feature_engineered_dataset.csv"

feature_df.to_csv(OUTPUT_FILE, index=False)

print("Feature-engineered dataset saved!")
print("Location:", OUTPUT_FILE)
print("Shape:", feature_df.shape)
print("File size (MB):", round(OUTPUT_FILE.stat().st_size / (1024 * 1024), 2))

Feature-engineered dataset saved!
Location: C:\Users\anshu\Desktop\ML project\aqi-intelligence\aqi-intelligence\datanew\aqi_rebuild\data\feature_engineered_dataset.csv
Shape: (197697, 59)
File size (MB): 86.02
